# 1. Studies: ketepatan perhitungan lokal
Tujuan: memeriksa rumus deterministik terhadap referensi independen dan memisahkannya dari coverage data. Ini validasi deskriptif, bukan akurasi prediksi saham. Notebook belum dijalankan; pengguna mengendalikan eksekusinya.

In [ ]:
OBJECTIVE = 'Reference correctness and local-source coverage'
print(OBJECTIVE)

## 2. Imports dan lingkungan
Hanya runtime lokal dan paket proyek. Tidak ada API eksternal, training, atau akses key. Seed tidak diperlukan untuk rumus deterministik.

In [ ]:
import os, sys, json, math, platform
from pathlib import Path
from datetime import datetime, timezone
print('Python:', platform.python_version())
ATOL, RTOL = 1e-10, 1e-8

## 3. Guard Colab/Kaggle/local
Colab: unggah folder proyek berisi src/idx_evidence_lab, data/raw/sectors beserta metadata, dan docs/prototypes/portfolio-factor-zoo-data.json ke /content/idx-evidence-lab. Kaggle: tambahkan Dataset pribadi berisi struktur tersebut, misalnya /kaggle/input/idx-evidence-lab-local; set STUDIES_PROJECT_ROOT ke lokasi attachment sebenarnya. Tidak memakai path komputer pengembang atau fallback jaringan.

In [ ]:
candidate = os.environ.get('STUDIES_PROJECT_ROOT')
if candidate:
    ROOT = Path(candidate)
elif Path('/kaggle').exists():
    ROOT = Path('/kaggle/input/idx-evidence-lab-local')
elif 'google.colab' in sys.modules:
    ROOT = Path('/content/idx-evidence-lab')
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT/'src/idx_evidence_lab').is_dir() or not (ROOT/'data/raw/sectors').is_dir():
    raise RuntimeError('Lampirkan folder src dan snapshot Sectors beserta metadata; set STUDIES_PROJECT_ROOT. Tidak ada fallback eksternal.')
sys.path.insert(0, str(ROOT/'src'))
from idx_evidence_lab.studies_transforms import transform_series
from idx_evidence_lab.studies_registry import parse_study_request, WIDGETS
from idx_evidence_lab.studies_service import run_study
from idx_evidence_lab.market_data import load_lq45_universe, load_issuer_daily
print('Project attachments:', ROOT)

## 4. Loading dan inspeksi satu per satu
Universe berasal dari snapshot kini, bukan membership historis. Daily loader mengecek provider, ticker, metadata dan hash. AADI dapat memiliki histori lebih pendek.

In [ ]:
universe = load_lq45_universe(ROOT)
tickers = [r['ticker'] for r in universe['symbols']]
print({k: universe.get(k) for k in ('symbol_count','data_quality','source_file','issues')})
snapshots = {t: load_issuer_daily(ROOT,t) for t in tickers}
for t,data in snapshots.items():
    print(t, data['data_quality'], data['observation_count'], data['coverage_start'], data['coverage_end'], data['source_files'])

## 5. Audit schema dan kualitas
Missing tidak diisi nol. Tanggal duplikat/konflik dan hash bermasalah menahan perhitungan. Volume/harga/flow mempertahankan unitnya.

In [ ]:
quality = [{'ticker':t,'quality':d['data_quality'],'rows':len(d['series']),'issues':d['issues']} for t,d in snapshots.items()]
for row in quality: print(row)

## 6. EDA untuk coverage
Ringkasan panjang histori menjelaskan keterbatasan warm-up; bukan ukuran ketepatan rumus. Visual per seri tersedia pada workspace Studies.

In [ ]:
lengths = sorted((len(d['series']),t) for t,d in snapshots.items())
print('Shortest/longest:', lengths[:5], lengths[-5:])
print('Verified daily:', sum(d['data_quality']=='VERIFIED' for d in snapshots.values()), '/', len(tickers))

## 7. Formula dan fixture referensi
Difference = x[t]-x[t-1]; percent change = x[t]/x[t-1]-1 untuk field positif; mean trailing W; std sample ddof=1; robust z = (x-median)/(1.4826*MAD). Expected di bawah dihitung tangan, tidak dari helper produksi.

In [ ]:
dates=['2026-01-02','2026-01-05','2026-01-06']
fixture=[{'date':d,'close':v} for d,v in zip(dates,[1.,2.,3.])]
references=[('difference',2,[None,1.,1.]),('pct_change',2,[None,1.,.5]),('rolling_mean',2,[None,1.5,2.5]),('rolling_std',2,[None,.7071067811865476,.7071067811865476]),('robust_z',3,[None,None,.6744907594765952])]
print(fixture)
print(references)

## 8. Cutoff dan leakage guard
Tidak ada fitting/split ML. Prefix invariance memeriksa bahwa menambah observasi masa depan tidak mengubah nilai historis. Window mengikuti kalender yang dinyatakan dan tidak menghapus gap.

In [ ]:
future=fixture+[{'date':'2026-01-07','close':100.}]
for transform,w,_ in references:
    a=transform_series(fixture,field='close',transform=transform,window=w,calendar=dates,unit='IDR')
    b=transform_series(future,field='close',transform=transform,window=w,calendar=dates+['2026-01-07'],unit='IDR')
    assert a['series']==b['series'][:3], transform
print('Prefix invariance PASS')

## 9. Baseline independen
Baseline adalah tabel expected hand-calculated di bagian 7, bukan model prediksi. Nilai nol standar deviasi pada seri konstan sah; MAD nol menghasilkan null.

In [ ]:
constant=[{'date':d,'close':2.} for d in dates]
assert transform_series(constant,field='close',transform='rolling_std',window=2,calendar=dates,unit='IDR')['series'][-1]['value']==0
assert transform_series(constant,field='close',transform='robust_z',window=3,calendar=dates,unit='IDR')['series'][-1]['reason']=='ZERO_MAD'
print('Constant baseline PASS')

## 10. Definisi engine
Registry tertutup memiliki 16 panel. Executor memakai snapshot/engine lokal; tidak menerima kode bebas. Portfolio ratios/CAPM memerlukan artifact Portfolio Lab yang sesuai, bukan bobot buatan.

In [ ]:
print('Widget allowlist:', WIDGETS)
print('Transform version: custom-trailing-v1-ddof1; numerical tolerances:', ATOL, RTOL)

## 11. Training tidak berlaku
Tidak ada model yang dilatih, probabilitas forecast, hyperparameter tuning, atau backtest panjang. Tidak menyebut analisis ini trained atau predictive accuracy.

In [ ]:
training_status='NOT_APPLICABLE_DETERMINISTIC_ENGINE'
print(training_status)

## 12. Validasi referensi per nilai
Expected, actual, selisih dan verdict ditampilkan satu per satu. Seluruh kasus mandatory harus lolos; aggregate 85% tidak membenarkan satu rumus kritis salah.

In [ ]:
verdicts=[]
for transform,w,expected in references:
    result=transform_series(fixture,field='close',transform=transform,window=w,calendar=dates,unit='IDR')
    for point,want in zip(result['series'],expected):
        value=point['value']
        passed=value is None if want is None else value is not None and math.isclose(value,want,rel_tol=RTOL,abs_tol=ATOL)
        record={'feature':transform,'date':point['date'],'expected':want,'actual':value,'difference':None if value is None or want is None else value-want,'passed':passed}
        verdicts.append(record);print(record)
assert all(v['passed'] for v in verdicts), 'Mandatory formula failure blocks acceptance'
reference_correctness=sum(v['passed'] for v in verdicts)/len(verdicts)
print('reference_correctness:',reference_correctness)

## 13. Hasil dan coverage lokal
Setiap panel BBCA dijalankan offline. Status partial/insufficient adalah hasil jujur, bukan error yang diubah menjadi angka. Coverage rolling mean diuji terpisah untuk seluruh emiten.

In [ ]:
target='BBCA' if 'BBCA' in tickers else tickers[0]
widget_results=[]
for widget in WIDGETS:
    request=parse_study_request({'widget_id':widget,'targets':[target]},known_tickers=tickers,sectors={})
    result=run_study(ROOT,request);widget_results.append(result)
    print(widget,result['status'],result['coverage'],result['warnings'])
issuer_coverage=[]
for ticker in tickers:
    request=parse_study_request({'widget_id':'custom_engineering','targets':[ticker],'params':{'transform':'rolling_mean'}},known_tickers=tickers,sectors={})
    result=run_study(ROOT,request)
    issuer_coverage.append({'ticker':ticker,'status':result['status'],'sources':len(result['sources'])})
print('issuer_coverage:',issuer_coverage)

## 14. Interpretasi, bukan trading
Output mendukung riset. Event detected berbeda dengan sampel matang 5D/20D. Tidak memperbaiki 10/9 menjadi 10/10, tidak mengubah optimizer atau melakukan order.

In [ ]:
events=next(r for r in widget_results if r['widget_id']=='event_outcomes')
print('Event counts and evidence:',events['tables'][:3])

## 15. Kesimpulan dan keterbatasan
Reference correctness hanya mengukur kasus rumus yang diuji. Coverage data, integrasi UI, rasio lintas sektor dan adjustment corporate actions adalah dimensi terpisah. Ini bukan bukti akurasi forecast atau alpha.

In [ ]:
print({'reference_correctness':reference_correctness,'reference_cases':len(verdicts),'issuer_coverage':len(issuer_coverage),'training':training_status,'forecast_accuracy':'NOT_MEASURED'})

## 16. Manifest reproducibility
Manifest berada dalam variabel notebook, tidak menyimpan key atau dokumen privat. User dapat mengekspor sendiri setelah meninjau. Run time UTC dicatat secara eksplisit.

In [ ]:
manifest={'run_time_utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),'formula_versions':sorted({r['formula_version'] for r in widget_results}),'reference_correctness':reference_correctness,'reference_cases':len(verdicts),'issuer_coverage':issuer_coverage,'source_hashes':{s['id']:s['sha256'] for r in widget_results for s in r['sources']},'network_calls':0,'training':training_status,'verification_scope':'Five custom transforms on explicit reference fixture; not all market features'}
print(json.dumps(manifest,indent=2,ensure_ascii=False))